# PARTE 2: Variables Espaciales
## Riesgo de Contaminación por Nitratos | La Rioja, 2015 - 2025

*Objetivo:* Construir las variables espaciales estáticas (suelo, uso del suelo, SIGPAC, zona vulnerable y altitud) para cada uno de los 104 puntos de muestreo de nitratos, como atributos del punto.

In [1]:
!pip install -q requests pandas geopandas owslib rasterio numpy shapely tqdm openpyxl pyogrio

In [3]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.merge import merge
from tqdm import tqdm

In [4]:
BASE_DIR   = Path(r'C:\Users\mcangulo\OneDrive - FEDERACION DE EMPRESAS DE LA RIOJA\Escritorio\dataset_larioja')
CHE_DIR    = BASE_DIR / '1_nitratos_che'
SOIL_DIR   = BASE_DIR / '3_suelo_soilgrids'
CORINE_DIR = BASE_DIR / '4_uso_suelo_corine'
SIGPAC_DIR = BASE_DIR / '5_sigpac'
ZONAS_DIR  = BASE_DIR / '6_zonas_vulnerables'
DEM_DIR    = BASE_DIR / '7_altitud_dem'
OUT_DIR    = BASE_DIR / '9_dataset_final'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# BBOX La Rioja EPSG:4326 
LAT_MIN, LAT_MAX = 41.90, 42.62
LON_MIN, LON_MAX = -3.05, -1.62

# BBOX en UTM30 ETRS89 (para filtros por x_utm30/y_utm30)
UTM_X_MIN, UTM_X_MAX = 480_000, 640_000
UTM_Y_MIN, UTM_Y_MAX = 4_630_000, 4_730_000

CRS_WGS84 = 'EPSG:4326'
CRS_UTM30 = 'EPSG:25830'

In [5]:
def buscar_archivo_vectorial(carpeta):
    for pat in ['*.gpkg','*.shp','*.geojson','*.json']:
        hits = sorted(Path(carpeta).rglob(pat))
        if hits: return hits[0]
    raise FileNotFoundError(f'Sin archivo vectorial en {carpeta}')

def print_check(label, ok, detalle=''):
    print(f"{'OK' if ok else 'FALLO'} {label}: {detalle}")

def validar_rango(df, col, minimo, maximo):
    if col not in df.columns:
        print_check(f'Rango {col}', False, 'columna no existe'); return
    mn, mx, nul = df[col].min(), df[col].max(), df[col].isna().sum()
    print_check(f'Rango {col}', mn >= minimo and mx <= maximo,
                f'min={mn:.3f}, max={mx:.3f}, nulos={nul}')

def resumen_nulos(df, top=30):
    n = df.isna().sum().sort_values(ascending=False).head(top)
    n = n[n > 0]
    return pd.DataFrame({'columna': n.index, 'n_nulos': n.values,
                         'pct': (100*n.values/len(df)).round(1)})

### 1. Construcción de puntos RNIT

Se parte directamente del CSV generado por la Parte 1, en lugar de volver a cargar el shapefile original de RNIT y filtrarlo de nuevo por el bounding box de La Rioja. Hacerlo así garantiza que ambas partes trabajen exactamente sobre los mismos puntos, evitando que aplicar el mismo filtro dos veces de forma independiente introduzca pequeñas diferencias entre P1 y P2. La clave `ipa` conecta ambas partes y debe coincidir exactamente entre ellas.

In [7]:
ruta_p1 = CHE_DIR / 'nitratos_larioja_2015_2025.csv'
if not ruta_p1.exists():
    raise FileNotFoundError(f'Output de P1 no encontrado: {ruta_p1}')

df_p1 = pd.read_csv(ruta_p1)
print(f'P1 cargado: {df_p1.shape[0]:,} filas x {df_p1.shape[1]} columnas')
print('IPAs unicos en P1:', df_p1['ipa'].nunique())

P1 cargado: 1,300 filas x 19 columnas
IPAs unicos en P1: 104


In [8]:
# Filtro de seguridad: por si la descarga de P1 incluyera puntos fuera de La Rioja
# Intento 1: columna provincia
if 'provincia' in df_p1.columns:
    mask_prov = df_p1['provincia'].str.upper().str.contains('RIOJA', na=False)
    print(f'Filas provincia=La Rioja: {mask_prov.sum():,}')
else:
    mask_prov = pd.Series(False, index=df_p1.index)
    print('Sin columna provincia')

# Intento 2: coordenadas UTM
if {'x_utm30','y_utm30'}.issubset(df_p1.columns):
    mask_utm = (df_p1['x_utm30'].between(UTM_X_MIN, UTM_X_MAX) &
                df_p1['y_utm30'].between(UTM_Y_MIN, UTM_Y_MAX))
    print(f'Filas dentro de bbox UTM La Rioja: {mask_utm.sum():,}')
else:
    mask_utm = pd.Series(True, index=df_p1.index)

mask_final = mask_prov & mask_utm if mask_prov.sum() > 0 else mask_utm
df_lr = df_p1[mask_final].copy()

n_total = df_p1['ipa'].nunique()
n_fuera = df_p1[~mask_final]['ipa'].nunique()
n_lr    = df_lr['ipa'].nunique()
print(f'IPAs totales en P1:     {n_total}')
print(f'IPAs fuera La Rioja:    {n_fuera}  <- eliminados')
print(f'IPAs en La Rioja:       {n_lr}  <- usados en P2')
if n_fuera > 0:
    print('AVISO: P1 contenia puntos fuera de La Rioja.')
    print('  Esto ocurre cuando la descarga CHE no filtra por provincia.')

Filas provincia=La Rioja: 1,300
Filas dentro de bbox UTM La Rioja: 1,300
IPAs totales en P1:     104
IPAs fuera La Rioja:    0  <- eliminados
IPAs en La Rioja:       104  <- usados en P2


In [9]:
# Tabla de puntos unicos (1 fila por IPA)
puntos = (
    df_lr[['ipa','x_utm30','y_utm30']]
    .drop_duplicates('ipa')
    .dropna(subset=['x_utm30','y_utm30'])
    .reset_index(drop=True)
)
# Añadir columnas descriptivas
for col in ['nombre_punto','municipio','provincia','masa_agua','red']:
    if col in df_lr.columns:
        puntos = puntos.merge(df_lr[['ipa',col]].drop_duplicates('ipa'), on='ipa', how='left')

gdf_larioja = gpd.GeoDataFrame(
    puntos,
    geometry=gpd.points_from_xy(puntos['x_utm30'], puntos['y_utm30']),
    crs=CRS_UTM30
)
gdf_wgs = gdf_larioja.to_crs(CRS_WGS84)
gdf_larioja['lon'] = gdf_wgs.geometry.x
gdf_larioja['lat'] = gdf_wgs.geometry.y

print(f'Puntos unicos La Rioja: {len(gdf_larioja)}')
print(f'CRS: {gdf_larioja.crs}')
print(f'Bbox lon: [{gdf_larioja.lon.min():.3f}, {gdf_larioja.lon.max():.3f}]')
print(f'Bbox lat: [{gdf_larioja.lat.min():.3f}, {gdf_larioja.lat.max():.3f}]')
display(gdf_larioja[['ipa','x_utm30','y_utm30','lon','lat']].head(8))

Puntos unicos La Rioja: 104
CRS: EPSG:25830
Bbox lon: [-3.088, -1.697]
Bbox lat: [41.963, 42.626]


,ipa,x_utm30,y_utm30,lon,lat
0,211020172,497981,4705042,-3.024571,42.497736
1,220950053,513179,4709814,-2.839502,42.540602
2,210980206,510029,4712494,-2.877816,42.564785
3,210980114,509312,4713861,-2.886529,42.577105
4,210980017,510361,4707537,-2.873861,42.520139
5,210980214,507946,4710298,-2.903224,42.545033
6,211040003,509763,4704680,-2.881190,42.494417
7,211040524,511803,4705137,-2.856355,42.498504


### 2. Construcción de variable Nº3: SOILGRIDS

Se detectó y corrigió un error en el factor de conversión de las variables de capacidad hídrica del suelo (`wv0033` y `wv1500`): el factor original (0,1) daba valores físicamente imposibles, mayores a 1 cm³/cm³; el factor correcto, según la documentación de ISRIC, es 0,001. Además, cada función de extracción abre el ráster una sola vez con `with rasterio.open()`, en vez de reabrirlo por cada variable, lo que mejora el rendimiento al procesar los 24 archivos de SoilGrids.

In [10]:
# Factores de conversion verificados contra la documentacion oficial de ISRIC (ver nota de arriba)

SOIL_FACTORES = {
    'clay':0.1,'silt':0.1,'sand':0.1,'phh2o':0.1,'cec':0.1,'soc':0.1,
    'bdod':0.01,'nitrogen':0.01,'cfvo':0.1,
    'wv0033':0.001, 
    'wv1500':0.001,  
}
RANGOS_VALIDOS = {
    'clay':(0,100),'silt':(0,100),'sand':(0,100),
    'phh2o':(2,12),'cec':(0,300),'soc':(0,200),
    'bdod':(0,2.5),'wv0033':(0,1),'wv1500':(0,1),'cfvo':(0,100),
}

def detectar_variable_soil(nombre):
    n = nombre.lower()
    return next((v for v in SOIL_FACTORES if v in n), None)

def nombre_columna_soil(ruta):
    stem = ruta.stem.lower().replace('_mean','').replace('_q0.5','').replace('-','_')
    return 'soil_' + stem

# Una sola apertura del raster por funcion (eficiencia)

def extraer_raster_en_puntos(gdf, ruta, nombre_col, factor=1.0):
    NODATA_VALS = {-32768,-32767,32767,-9999}
    with rasterio.open(ruta) as src:
        gdf_t = gdf.to_crs(src.crs)
        coords = [(g.x,g.y) for g in gdf_t.geometry]
        nd = src.nodata
        vals = []
        for v in src.sample(coords):
            x = float(v[0])
            if (nd is not None and np.isclose(x,nd)) or x in NODATA_VALS:
                vals.append(np.nan)
            else:
                vals.append(x*factor)
    return pd.Series(vals, name=nombre_col)

def extraer_soilgrids(gdf, soil_dir=SOIL_DIR):
    tifs = sorted(soil_dir.rglob('*.tif'))
    if not tifs: raise FileNotFoundError(f'Sin TIFFs en {soil_dir}')
    df = gdf[['ipa','lat','lon']].copy().reset_index(drop=True)
    resumen = []
    for tif in tqdm(tifs, desc='SoilGrids'):
        var = detectar_variable_soil(tif.name)
        if var is None: continue
        col, factor = nombre_columna_soil(tif), SOIL_FACTORES[var]
        serie = extraer_raster_en_puntos(gdf, tif, col, factor)
        
        # Validar rango fisico
        if var in RANGOS_VALIDOS:
            rmin,rmax = RANGOS_VALIDOS[var]
            fuera = (~serie.isna())&((serie<rmin)|(serie>rmax))
            serie = serie.where(~fuera, np.nan)
        df[col] = serie.values
        resumen.append({'archivo':tif.name,'columna':col,'factor':factor,
            'min':df[col].min(),'max':df[col].max(),
            'nulos':df[col].isna().sum(),'ceros':int((df[col]==0).sum())})
    return df, pd.DataFrame(resumen)

In [11]:

df_soil, resumen_soil = extraer_soilgrids(gdf_larioja)
print(f'Shape SoilGrids: {df_soil.shape}')
display(resumen_soil)

cols_soil = [c for c in df_soil.columns if c.startswith('soil_')]
pct_ceros = (df_soil[cols_soil]==0).mean().max()*100
print('OK sin exceso de ceros' if pct_ceros < 80 else f'AVISO: max ceros {pct_ceros:.1f}%')

# Agua disponible
col033 = next((c for c in df_soil.columns if 'wv0033' in c and '0-5' in c), None)
col150 = next((c for c in df_soil.columns if 'wv1500' in c and '0-5' in c), None)
if col033 and col150:
    df_soil['agua_disponible_cm3cm3'] = df_soil[col033] - df_soil[col150]
    print(f'Agua disponible 0-5cm media: {df_soil.agua_disponible_cm3cm3.mean():.3f} cm3/cm3')
    print('  (rango esperado 0.05-0.25 para suelos La Rioja)')

# Verificacion del factor de wv0033: debe quedar en cm3/cm3 (<=1.0), no en porcentaje
if col033:
    mx = df_soil[col033].max()
    if mx > 1.0:
        print(f'ERROR: {col033} max={mx:.2f} > 1.0 — el factor sigue siendo incorrecto')
    else:
        print(f'OK: {col033} max={mx:.4f} — en rango correcto (<=1.0 cm3/cm3)')

display(df_soil.head(5))

SoilGrids: 100%|██████████| 24/24 [00:00<00:00, 26.35it/s]

Shape SoilGrids: (104, 27)


,archivo,columna,factor,min,max,nulos,ceros
0,cec_0-5cm.tif,soil_cec_0_5cm,0.100,0.0,23.600,0,7
1,cec_15-30cm.tif,soil_cec_15_30cm,0.100,0.0,20.500,0,7
2,cec_5-15cm.tif,soil_cec_5_15cm,0.100,0.0,21.200,0,7
3,clay_0-5cm.tif,soil_clay_0_5cm,0.100,0.0,31.000,0,7
4,clay_15-30cm.tif,soil_clay_15_30cm,0.100,0.0,35.300,0,7
5,clay_5-15cm.tif,soil_clay_5_15cm,0.100,0.0,31.300,0,7
6,phh2o_0-5cm.tif,soil_phh2o_0_5cm,0.100,6.0,8.100,7,0
7,phh2o_15-30cm.tif,soil_phh2o_15_30cm,0.100,6.1,8.200,7,0
8,phh2o_5-15cm.tif,soil_phh2o_5_15cm,0.100,6.1,8.200,7,0
9,sand_0-5cm.tif,soil_sand_0_5cm,0.100,0.0,49.000,0,7


OK sin exceso de ceros


,ipa,lat,lon,soil_cec_0_5cm,soil_cec_15_30cm,soil_cec_5_15cm,soil_clay_0_5cm,soil_clay_15_30cm,soil_clay_5_15cm,soil_phh2o_0_5cm,...,soil_silt_5_15cm,soil_soc_0_5cm,soil_soc_15_30cm,soil_soc_5_15cm,soil_wv0033_0_5cm,soil_wv0033_15_30cm,soil_wv0033_5_15cm,soil_wv1500_0_5cm,soil_wv1500_15_30cm,soil_wv1500_5_15cm
0,211020172,42.497736,-3.024571,19.1,15.8,17.5,23.2,24.9,25.1,7.7,...,48.4,27.6,12.1,12.7,0.347,0.337,0.340,0.180,0.185,0.183
1,220950053,42.540602,-2.839502,22.0,15.7,16.8,21.3,27.0,23.3,7.7,...,52.2,28.5,11.4,11.4,0.349,0.330,0.342,0.171,0.182,0.183
2,210980206,42.564785,-2.877816,20.1,16.3,18.5,20.0,28.9,23.5,7.7,...,43.1,28.2,12.0,12.3,0.345,0.330,0.340,0.167,0.187,0.179
3,210980114,42.577105,-2.886529,22.0,19.1,21.0,18.9,27.4,22.3,7.6,...,38.0,27.6,13.0,12.7,0.356,0.337,0.348,0.161,0.169,0.167
4,210980017,42.520139,-2.873861,19.3,16.0,16.8,20.5,23.8,21.7,7.8,...,52.6,24.1,11.0,11.4,0.338,0.324,0.332,0.178,0.187,0.187


### 3. Construcción de la variable Nº4: CORINE Land Cover 2012 / 2018


In [12]:
ruta_clc2012 = CORINE_DIR/'datos_suelo_2012'/'DATA'/'U2018_CLC2012_V2020_20u1.tif'
ruta_clc2018 = CORINE_DIR/'datos_suelo_2018'/'DATA'/'U2018_CLC2018_V2020_20u1.tif'

CLC_RASTER_A_CODE = {
    1:111,2:112,3:121,4:122,5:123,6:124,7:131,8:132,9:133,10:141,11:142,
    12:211,13:212,14:213,15:221,16:222,17:223,18:231,19:241,20:242,21:243,22:244,
    23:311,24:312,25:313,26:321,27:322,28:323,29:324,30:331,31:332,32:333,
    33:334,34:335,35:411,36:412,37:421,38:422,39:423,40:511,41:512,42:521,43:522,44:523,
}
CLC_DESC = {
    111:'Tejido urbano continuo',112:'Tejido urbano discontinuo',
    121:'Zonas industriales',141:'Zonas verdes urbanas',
    211:'Tierras labor secano',212:'Terrenos regados',213:'Arrozales',
    221:'Vinedos',222:'Frutales',223:'Olivares',231:'Praderas',
    241:'Cultivos anuales+perm',242:'Mosaico cultivos',
    243:'Agricultura+veg natural',244:'Sistemas agroforestales',
    311:'Bosque frondosas',312:'Bosque coniferas',313:'Bosque mixto',
    321:'Pastizales',322:'Matorrales',323:'Veg esclerofia',
    324:'Matorral boscoso',331:'Roquedo',332:'Roquedo',
    333:'Veg escasa',511:'Cursos de agua',512:'Laminas de agua',
}
AGRICOLA_CLC = {211,212,213,221,222,223,231,241,242,243,244}
CLC_NODATA   = -128

# Una sola apertura del raster 

def extraer_corine_puntos(gdf, ruta, anio):
    if not ruta.exists(): raise FileNotFoundError(f'Sin CLC {anio}: {ruta}')
    with rasterio.open(ruta) as src:
        gdf_t  = gdf.to_crs(src.crs)
        coords = [(g.x,g.y) for g in gdf_t.geometry]
        vals   = []
        for v in src.sample(coords):
            vi = int(v[0])
            vals.append(CLC_RASTER_A_CODE.get(vi,np.nan) if vi not in {CLC_NODATA,0} and 1<=vi<=44 else np.nan)
    df = pd.DataFrame({'ipa':gdf['ipa'].values, f'clc{anio}_codigo':vals})
    df[f'clc{anio}_nombre']   = df[f'clc{anio}_codigo'].map(CLC_DESC)
    df[f'clc{anio}_agricola'] = df[f'clc{anio}_codigo'].isin(AGRICOLA_CLC)
    return df

df_clc2012 = extraer_corine_puntos(gdf_larioja, ruta_clc2012, 2012)
df_clc2018 = extraer_corine_puntos(gdf_larioja, ruta_clc2018, 2018)
df_corine  = df_clc2012.merge(df_clc2018, on='ipa', how='outer')
df_corine['cambio_clc_2012_2018'] = df_corine['clc2012_codigo'] != df_corine['clc2018_codigo']

print(f'Shape CORINE: {df_corine.shape}')
print(f'Puntos agricolas 2018: {df_corine.clc2018_agricola.sum()} / {len(df_corine)}')
display(df_corine['clc2018_nombre'].value_counts(dropna=False).head(12))
display(df_corine.head(5))

Shape CORINE: (104, 8)
Puntos agricolas 2018: 72 / 104


clc2018_nombre
Tierras labor secano         23
Mosaico cultivos             19
Vinedos                      13
Terrenos regados             10
Bosque frondosas              7
Veg esclerofia                7
Frutales                      6
Matorrales                    4
Matorral boscoso              4
Tejido urbano discontinuo     3
Tejido urbano continuo        3
Praderas                      1
Name: count, dtype: int64

,ipa,clc2012_codigo,clc2012_nombre,clc2012_agricola,clc2018_codigo,clc2018_nombre,clc2018_agricola,cambio_clc_2012_2018
0,210920002,311,Bosque frondosas,False,311,Bosque frondosas,False,False
1,210940011,311,Bosque frondosas,False,311,Bosque frondosas,False,False
2,210960027,112,Tejido urbano discontinuo,False,112,Tejido urbano discontinuo,False,False
3,210970015,211,Tierras labor secano,True,211,Tierras labor secano,True,False
4,210970033,221,Vinedos,True,221,Vinedos,True,False


### 4. Construcción de la variable Nº5: SIGPAC

La unión espacial con las parcelas SIGPAC se hace con `sjoin_nearest` (distancia máxima 100 m) en lugar de una intersección estricta (`predicate='within'`). El motivo es que varios puntos de muestreo caían justo en el borde de una parcela y la intersección estricta los dejaba sin asignar; con un margen de 100 m se recuperan esos casos sin introducir asignaciones erróneas a parcelas lejanas.

In [13]:
SIGPAC_USOS = {
    'AG':'Agua','CA':'Viales','CI':'Citricos','ED':'Edificaciones',
    'FO':'Forestal','FS':'Frutos secos','FY':'Frutales','IM':'Improductivo',
    'IV':'Invernaderos','OV':'Olivar','PA':'Pasto arbolado','PR':'Pasto arbustivo',
    'PS':'Pastizal','TA':'Tierra arable','TH':'Huerta','VI':'Vinedo',
    'VO':'Vinedo-olivar','ZU':'Zona urbana','ZV':'Zona censurada',
    'MT':'Monte bajo / Matorral',
}
USOS_AGR_SIGPAC = {'TA','TH','VI','FY','CI','OV','FS','PS','PA','PR'}

def cargar_sigpac(sigpac_dir=SIGPAC_DIR):
    ruta_l = sigpac_dir/'sigpac_larioja_limpio.gpkg'
    ruta_r = sigpac_dir/'sigpac_larioja_raw.gpkg'
    if ruta_l.exists():   gdf = gpd.read_file(ruta_l); print('SIGPAC limpio:', ruta_l)
    elif ruta_r.exists(): gdf = gpd.read_file(ruta_r); print('SIGPAC raw:', ruta_r)
    else:
        gpkgs = sorted(sigpac_dir.rglob('*.gpkg'))
        if not gpkgs: raise FileNotFoundError(f'Sin SIGPAC en {sigpac_dir}')
        gdf = pd.concat([gpd.read_file(p) for p in tqdm(gpkgs)], ignore_index=True)
        gdf = gpd.GeoDataFrame(gdf, geometry='geometry')
    gdf.columns = [c.lower().strip() for c in gdf.columns]
    cols = [c for c in ['municipio','poligono','parcela','recinto','dn_surface',
                         'pendiente_media','altitud','coef_regadio','uso_sigpac','geometry']
            if c in gdf.columns]
    gdf = gdf[cols].copy()
    for col in ['dn_surface','pendiente_media','altitud','coef_regadio']:
        if col in gdf.columns: gdf[col] = pd.to_numeric(gdf[col], errors='coerce')
    if 'pendiente_media' in gdf.columns:
        p95 = gdf['pendiente_media'].quantile(0.95)
        gdf['pendiente_media_pct'] = gdf['pendiente_media']/100 if p95>100 else gdf['pendiente_media']
        if p95>100: print('AVISO: pendiente SIGPAC reescalada /100')
    if 'uso_sigpac' in gdf.columns:
        gdf['uso_sigpac'] = gdf['uso_sigpac'].astype(str).str.upper().str.strip()
        gdf['uso_nombre'] = gdf['uso_sigpac'].map(SIGPAC_USOS).fillna('No clasificado')
        gdf['es_agricola']= gdf['uso_sigpac'].isin(USOS_AGR_SIGPAC)
    if gdf.crs is None: gdf = gdf.set_crs('EPSG:4258')
    return gdf

# sjoin_nearest en lugar de interseccion estricta 
def extraer_sigpac_puntos(gdf_pts, gdf_sig):
    puntos = gdf_pts.to_crs(gdf_sig.crs).copy()
    campos = [c for c in ['uso_sigpac','uso_nombre','dn_surface',
                           'pendiente_media_pct','coef_regadio','es_agricola','geometry']
              if c in gdf_sig.columns]
    joined = gpd.sjoin_nearest(puntos[['ipa','geometry']], gdf_sig[campos],
                                how='left', max_distance=100, distance_col='dist_sigpac_m')
    joined = joined.drop_duplicates('ipa', keep='first')
    joined = joined.drop(columns=['index_right','geometry'], errors='ignore')
    return joined.rename(columns={
        'uso_sigpac':'sigpac_uso','uso_nombre':'sigpac_uso_nombre',
        'dn_surface':'sigpac_superficie_m2','pendiente_media_pct':'sigpac_pendiente_pct',
        'coef_regadio':'sigpac_coef_regadio','es_agricola':'sigpac_es_agricola',})

gdf_sigpac       = cargar_sigpac()
df_sigpac_puntos = extraer_sigpac_puntos(gdf_larioja, gdf_sigpac)
print(f'SIGPAC: {len(gdf_sigpac):,} recintos | CRS: {gdf_sigpac.crs}')
print(f'Shape SIGPAC puntos: {df_sigpac_puntos.shape}')
if 'sigpac_pendiente_pct' in df_sigpac_puntos.columns:
    validar_rango(df_sigpac_puntos,'sigpac_pendiente_pct',0,300)
n_sin = df_sigpac_puntos['sigpac_uso'].isna().sum()
if n_sin > 0: print(f'AVISO: {n_sin} puntos sin recinto SIGPAC a <100m')
display(df_sigpac_puntos['sigpac_uso_nombre'].value_counts(dropna=False).head(10))
display(df_sigpac_puntos.head(5))

SIGPAC limpio: C:\Users\mcangulo\OneDrive - FEDERACION DE EMPRESAS DE LA RIOJA\Escritorio\dataset_larioja\5_sigpac\sigpac_larioja_limpio.gpkg
AVISO: pendiente SIGPAC reescalada /100
SIGPAC: 1,074,578 recintos | CRS: EPSG:4258
Shape SIGPAC puntos: (104, 8)
OK Rango sigpac_pendiente_pct: min=0.040, max=5.220, nulos=0


sigpac_uso_nombre
Monte bajo / Matorral    19
Improductivo             13
Tierra arable            12
Agua                     11
Forestal                 10
Viales                   10
Zona urbana              10
Vinedo                    6
Frutales                  3
Olivar                    3
Name: count, dtype: int64

,ipa,sigpac_uso,sigpac_uso_nombre,sigpac_superficie_m2,sigpac_pendiente_pct,sigpac_coef_regadio,sigpac_es_agricola,dist_sigpac_m
0,211020172,MT,Monte bajo / Matorral,1005.248869,2.29,0.0,False,0.0
1,220950053,MT,Monte bajo / Matorral,354.816679,1.19,0.0,False,0.0
2,210980206,TH,Huerta,73436.884930,0.39,100.0,True,0.0
3,210980114,TA,Tierra arable,8850.174446,0.18,0.0,True,0.0
4,210980017,MT,Monte bajo / Matorral,1133.337844,1.30,0.0,False,0.0


### 5. Construcción de la variable Nº6: Zonas Vulnerables

In [14]:
def cargar_zonas(zonas_dir=ZONAS_DIR):
    ruta = buscar_archivo_vectorial(zonas_dir)
    gdf  = gpd.read_file(ruta)
    if gdf.crs is None: gdf = gdf.set_crs(CRS_WGS84)
    gdf = gdf.to_crs(CRS_WGS84)
    gdf['geometry'] = gdf.geometry.buffer(0)
    return gdf

def marcar_zona_vulnerable(gdf_pts, gdf_zonas):
    puntos = gdf_pts.to_crs(gdf_zonas.crs).copy()
    joined = gpd.sjoin(puntos[['ipa','geometry']], gdf_zonas[['geometry']],
                       how='left', predicate='within')
    
    # Agrupar por ipa (no .values) para evitar desalineacion de indices
    mapeado = (joined.groupby('ipa')['index_right'].first().notna()
               .astype(bool).reset_index()
               .rename(columns={'index_right':'zona_vulnerable_nitratos'}))
    return mapeado

gdf_zonas       = cargar_zonas()
df_zonas_puntos = marcar_zona_vulnerable(gdf_larioja, gdf_zonas)
n_vul = df_zonas_puntos['zona_vulnerable_nitratos'].sum()
print(f'Zonas cargadas: {len(gdf_zonas)}')
print(f'Puntos en zona vulnerable: {n_vul} / {len(df_zonas_puntos)} ({100*n_vul/len(df_zonas_puntos):.1f}%)')
display(df_zonas_puntos.head(5))

Zonas cargadas: 16
Puntos en zona vulnerable: 41 / 104 (39.4%)


,ipa,zona_vulnerable_nitratos
0,210920002,False
1,210940011,True
2,210960027,True
3,210970015,True
4,210970033,True


### 6. Construcción de la variable Nº7: Altitud (DEM)

El rango físico válido se ajustó a 250–2.300 m, calibrado a la geografía real de La Rioja (263 m en Alfaro, el punto más bajo, hasta 2.271 m en el Pico Urbión, el más alto), en vez de usar un rango genérico que podría descartar valores legítimos en los extremos del territorio.

In [15]:
def obtener_dem(dem_dir=DEM_DIR):
    ruta_m = dem_dir/'dem_larioja_merge.tif'
    if ruta_m.exists(): return ruta_m
    tifs = sorted(dem_dir.rglob('*.tif'))
    if not tifs: raise FileNotFoundError(f'Sin TIFFs DEM en {dem_dir}')
    srcs = [rasterio.open(p) for p in tifs]
    mos, tr = merge(srcs)
    meta = srcs[0].meta.copy()
    meta.update(driver='GTiff',height=mos.shape[1],width=mos.shape[2],transform=tr,count=1)
    with rasterio.open(ruta_m,'w',**meta) as dst: dst.write(mos)
    for s in srcs: s.close()
    return ruta_m

# Rango calibrado a la geografia de La Rioja 
def extraer_altitud(gdf, ruta_dem, alt_min=250, alt_max=2300):
    ND = {-32768,-32767,-9999}
    with rasterio.open(ruta_dem) as src:
        gdf_t  = gdf.to_crs(src.crs)
        coords = [(g.x,g.y) for g in gdf_t.geometry]
        nd, vals = src.nodata, []
        for v in src.sample(coords):
            x = float(v[0])
            if (nd is not None and np.isclose(x,nd)) or x in ND or x<alt_min or x>alt_max:
                vals.append(np.nan)
            else:
                vals.append(x)
    return pd.DataFrame({'ipa':gdf['ipa'].values,'altitud_dem_m':vals})

ruta_dem = obtener_dem()
df_dem   = extraer_altitud(gdf_larioja, ruta_dem)
n_ok = df_dem['altitud_dem_m'].notna().sum()
print(f'DEM: {ruta_dem}')
print(f'Puntos con altitud: {n_ok} / {len(df_dem)}')
validar_rango(df_dem,'altitud_dem_m',250,2300)
if n_ok < 0.9*len(df_dem): print('AVISO: <90% con altitud — verificar cobertura DEM')
display(df_dem.head(5))

DEM: C:\Users\mcangulo\OneDrive - FEDERACION DE EMPRESAS DE LA RIOJA\Escritorio\dataset_larioja\7_altitud_dem\dem_larioja_merge.tif
Puntos con altitud: 104 / 104
OK Rango altitud_dem_m: min=267.652, max=1449.052, nulos=0


,ipa,altitud_dem_m
0,211020172,575.192017
1,220950053,501.440033
2,210980206,481.456024
3,210980114,463.660034
4,210980017,535.187683


### 7.  Integración de variables construidas 

Efectuada bajo la union por clave `ipa`

In [23]:
gdf_final = gdf_larioja.copy()

fuentes = [
    ('SoilGrids',         df_soil),
    ('CORINE',            df_corine),
    ('SIGPAC',            df_sigpac_puntos),
    ('Zonas vulnerables', df_zonas_puntos),
    ('DEM',               df_dem),
]

for nombre, df_aux in fuentes:
    n_antes = len(gdf_final)
    gdf_final = gdf_final.merge(df_aux, on='ipa', how='left')
    n_despues = len(gdf_final)
    print_check(f'Union {nombre}', n_antes==n_despues,
                f'filas antes={n_antes}, despues={n_despues}')

print(f'Shape final: {gdf_final.shape}')
display(gdf_final.drop(columns='geometry').head(4))

OK Union SoilGrids: filas antes=104, despues=104
OK Union CORINE: filas antes=104, despues=104
OK Union SIGPAC: filas antes=104, despues=104
OK Union Zonas vulnerables: filas antes=104, despues=104
OK Union DEM: filas antes=104, despues=104
Shape final: (104, 53)


,ipa,x_utm30,y_utm30,nombre_punto,municipio,provincia,masa_agua,red,lon_x,lat_x,...,cambio_clc_2012_2018,sigpac_uso,sigpac_uso_nombre,sigpac_superficie_m2,sigpac_pendiente_pct,sigpac_coef_regadio,sigpac_es_agricola,dist_sigpac_m,zona_vulnerable_nitratos,altitud_dem_m
0,211020172,497981,4705042,MANANTIAL DEL CAÑO,HERRAMÉLLURI,Rioja (La),ALUVIAL DEL TIRÓN,RNIT,-3.024571,42.497736,...,False,MT,Monte bajo / Matorral,1005.248869,2.29,0.0,False,0.0,True,575.192017
1,220950053,513179,4709814,MANANTIAL DE OLLONGUI,RODEZNO,Rioja (La),ALUVIAL DEL OJA,RNIT,-2.839502,42.540602,...,False,MT,Monte bajo / Matorral,354.816679,1.19,0.0,False,0.0,True,501.440033
2,210980206,510029,4712494,FUENTE DEL ESTRECHO,HARO,Rioja (La),ALUVIAL DEL OJA,RNIT,-2.877816,42.564785,...,False,TH,Huerta,73436.884930,0.39,100.0,True,0.0,True,481.456024
3,210980114,509312,4713861,MALZAPATO,HARO,Rioja (La),ALUVIAL DEL OJA,RNIT,-2.886529,42.577105,...,False,TA,Tierra arable,8850.174446,0.18,0.0,True,0.0,True,463.660034


In [24]:
print('================ CONTROLES FINALES ================')
print_check('Num. puntos La Rioja', len(gdf_final)==len(gdf_larioja), f'{len(gdf_final)} filas')
print_check('ipa unico', gdf_final['ipa'].is_unique, f'dup={gdf_final.ipa.duplicated().sum()}')

for col in ['clc2012_codigo','clc2018_codigo']:
    if col in gdf_final.columns:
        print_check(f'{col}', gdf_final[col].notna().mean()>0.95, f'nulos={gdf_final[col].isna().sum()}')
    else:
        print_check(f'{col}', False, 'columna no existe')

cols_soil_f = [c for c in gdf_final.columns if c.startswith('soil_')]
if cols_soil_f:
    max_nul = gdf_final[cols_soil_f].isna().mean().max()*100
    max_cer = (gdf_final[cols_soil_f]==0).mean().max()*100
    print_check('SoilGrids nulos', max_nul<50, f'max={max_nul:.1f}%')
    print_check('SoilGrids ceros', max_cer<80, f'max={max_cer:.1f}%')
    col_wv = next((c for c in gdf_final.columns if 'wv0033' in c), None)
    if col_wv:
        mx = gdf_final[col_wv].max()
        print_check(f'C3: {col_wv} en cm3/cm3', mx<=1.0, f'max={mx:.4f} (debe ser <=1.0)')

if 'sigpac_pendiente_pct' in gdf_final.columns:
    validar_rango(gdf_final,'sigpac_pendiente_pct',0,300)
if 'zona_vulnerable_nitratos' in gdf_final.columns:
    n_zv = int(gdf_final['zona_vulnerable_nitratos'].sum())
    print_check('Zonas vulnerables', True, f'{n_zv}/{len(gdf_final)} puntos')
if 'altitud_dem_m' in gdf_final.columns:
    print_check('DEM cobertura >90%', gdf_final['altitud_dem_m'].notna().mean()>0.9,
                f'nulos={gdf_final.altitud_dem_m.isna().sum()}')
    validar_rango(gdf_final,'altitud_dem_m',250,2300)

print('\nColumnas con nulos:')
display(resumen_nulos(gdf_final, top=20))

================ CONTROLES FINALES ================
OK Num. puntos La Rioja: 104 filas
OK ipa unico: dup=0
OK clc2012_codigo: nulos=0
OK clc2018_codigo: nulos=0
OK SoilGrids nulos: max=6.7%
OK SoilGrids ceros: max=6.7%
OK C3: soil_wv0033_0_5cm en cm3/cm3: max=0.3640 (debe ser <=1.0)
OK Rango sigpac_pendiente_pct: min=0.040, max=5.220, nulos=0
OK Zonas vulnerables: 41/104 puntos
OK DEM cobertura >90%: nulos=0
OK Rango altitud_dem_m: min=267.652, max=1449.052, nulos=0

Columnas con nulos:


,columna,n_nulos,pct
0,sigpac_coef_regadio,44,42.3
1,soil_phh2o_5_15cm,7,6.7
2,soil_phh2o_15_30cm,7,6.7
3,soil_phh2o_0_5cm,7,6.7
4,clc2012_nombre,1,1.0
5,clc2018_nombre,1,1.0


In [26]:
ruta_gpkg  = OUT_DIR/'puntos_rnit_larioja_variables_espaciales_limpio.gpkg'
ruta_csv   = OUT_DIR/'puntos_rnit_larioja_variables_espaciales_limpio.csv'
ruta_excel = OUT_DIR/'puntos_rnit_larioja_variables_espaciales_limpio.xlsx'

gdf_final.to_file(ruta_gpkg, layer='puntos', driver='GPKG')
df_tabla = pd.DataFrame(gdf_final.drop(columns='geometry', errors='ignore'))
df_tabla.to_csv(ruta_csv, index=False, encoding='utf-8-sig')
df_tabla.to_excel(ruta_excel, index=False)

print('OK Dataset espacial guardado')
print('GPKG: ', ruta_gpkg)
print('CSV:  ', ruta_csv)
print('Filas:', len(gdf_final), '| Columnas:', gdf_final.shape[1])
print()
print('VERIFICACION CLAVE IPA (critica para Parte 4):')
ipas_p1_lr = set(df_lr['ipa'].astype(str).unique())
ipas_p2    = set(gdf_final['ipa'].astype(str).unique())
comunes    = ipas_p1_lr & ipas_p2
print(f'  IPAs La Rioja en P1: {len(ipas_p1_lr)}')
print(f'  IPAs en P2 output:   {len(ipas_p2)}')
print(f'  IPAs comunes:        {len(comunes)}')
if len(comunes) == len(ipas_p2):
    print('  OK 100% match -> merge en Parte 4 dara 0 nulos por IPA')
else:
    print('  AVISO: match incompleto — revisar filtros')
print()

OK Dataset espacial guardado
GPKG:  C:\Users\mcangulo\OneDrive - FEDERACION DE EMPRESAS DE LA RIOJA\Escritorio\dataset_larioja\9_dataset_final\puntos_rnit_larioja_variables_espaciales_limpio.gpkg
CSV:   C:\Users\mcangulo\OneDrive - FEDERACION DE EMPRESAS DE LA RIOJA\Escritorio\dataset_larioja\9_dataset_final\puntos_rnit_larioja_variables_espaciales_limpio.csv
Filas: 104 | Columnas: 53

VERIFICACION CLAVE IPA (critica para Parte 4):
  IPAs La Rioja en P1: 104
  IPAs en P2 output:   104
  IPAs comunes:        104
  OK 100% match -> merge en Parte 4 dara 0 nulos por IPA



#### Resumen de verificaciones de calidad de datos en este notebook

A lo largo de la construcción de variables espaciales se incluyeron las siguientes verificaciones automáticas:

- Cobertura geográfica de P1: confirma que ningún punto de la Parte 1 quedó fuera de La Rioja.
- Exceso de ceros en SoilGrids: ninguna variable de suelo debe superar el 80% de valores en cero (indicaría un problema de extracción, no un dato real).
- Reescalado de pendiente SIGPAC: si la pendiente llega en una unidad distinta a la esperada, se reescala automáticamente y se avisa.
- Cobertura del modelo digital de elevaciones (DEM): más del 90% de los puntos deben tener altitud asignada.
- Coincidencia de claves IPA entre Parte 1 y Parte 2: el 100% de los puntos de Parte 1 debe tener su contraparte aquí, condición necesaria para que la unión de la Parte 4 no pierda observaciones.

En la ejecución de este notebook, ninguna de estas verificaciones se activó como advertencia: la cobertura fue del 100% en todos los casos, confirmando que las 104 variables espaciales se construyeron correctamente.